# Retail Sales Data Analysis using Pandas

**Project objective:** Demonstrate data analysis skills using Pandas by loading a CSV, inspecting and cleaning data, applying filtering/grouping/aggregation, and explaining business insights.

**Dataset:** `retail_sales_raw.csv` (synthetic practice dataset; 1,800 generated orders). Keep this CSV in the same folder as this notebook.

## 1. Import libraries and load the CSV

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("retail_sales_raw.csv")
print("Dataset shape:", df.shape)
display(df.head())

## 2. Inspect the dataset

In [ ]:
df.info()
display(df.describe(include="all").T)
print("Missing values by column:")
display(df.isna().sum())
print("Duplicate order IDs:", df["Order_ID"].duplicated().sum())

## 3. Clean missing and incorrect data
We parse dates and numeric columns, standardize text, remove duplicate orders, and exclude records with missing key fields or invalid non-positive quantity/price. Sales and profit are recalculated from the cleaned inputs.

In [ ]:
clean = df.copy()

clean["Order_Date"] = pd.to_datetime(clean["Order_Date"], errors="coerce")

text_cols = ["Region", "Category", "Product", "Customer_Segment"]
for col in text_cols:
    clean[col] = clean[col].astype("string").str.strip()

numeric_cols = ["Quantity", "Unit_Price", "Discount", "Unit_Cost", "Sales", "Profit"]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

clean.loc[clean["Quantity"] <= 0, "Quantity"] = np.nan
clean.loc[clean["Unit_Price"] <= 0, "Unit_Price"] = np.nan

clean = clean.drop_duplicates(subset="Order_ID")
required_cols = ["Order_Date", "Region", "Category", "Product",
                 "Customer_Segment", "Quantity", "Unit_Price", "Unit_Cost"]
clean = clean.dropna(subset=required_cols)

clean["Discount"] = clean["Discount"].fillna(0)
clean["Sales"] = (clean["Quantity"] * clean["Unit_Price"] *
                  (1 - clean["Discount"])).round(2)
clean["Profit"] = (clean["Sales"] -
                   clean["Quantity"] * clean["Unit_Cost"]).round(2)
clean["Month"] = clean["Order_Date"].dt.month_name()

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(clean))
print("Remaining missing values:", int(clean.isna().sum().sum()))
display(clean.head())

## 4. Filtering
Example: show higher-value orders (sales above ₹10,000).

In [ ]:
high_value_orders = clean[clean["Sales"] > 10000]
print("High-value orders:", len(high_value_orders))
display(high_value_orders[["Order_ID", "Region", "Category", "Sales", "Profit"]].head(10))

## 5. Grouping and aggregation
Compare sales and profit by category, region, and customer segment.

In [ ]:
category_summary = clean.groupby("Category").agg(
    Orders=("Order_ID", "count"),
    Total_Sales=("Sales", "sum"),
    Average_Sales=("Sales", "mean"),
    Total_Profit=("Profit", "sum")
).sort_values("Total_Sales", ascending=False)

region_summary = clean.groupby("Region").agg(
    Orders=("Order_ID", "count"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).sort_values("Total_Sales", ascending=False)

segment_summary = clean.groupby("Customer_Segment")["Sales"].sum().sort_values(ascending=False)

display(category_summary.round(2))
display(region_summary.round(2))
display(segment_summary.round(2))

## 6. Overall business metrics

In [ ]:
total_sales = clean["Sales"].sum()
total_profit = clean["Profit"].sum()
average_order_value = clean["Sales"].mean()
profit_margin = (total_profit / total_sales) * 100

print(f"Clean orders: {len(clean):,}")
print(f"Total sales: ₹{total_sales:,.2f}")
print(f"Total profit: ₹{total_profit:,.2f}")
print(f"Average order value: ₹{average_order_value:,.2f}")
print(f"Profit margin: {profit_margin:.2f}%")

## 7. Key insights
Use the tables above to identify the leading category, region, and customer segment. The generated dataset is synthetic, so findings are for demonstrating analysis methods and should not be presented as real company results.

In [ ]:
best_category = category_summary.index[0]
best_region = region_summary.index[0]
best_segment = segment_summary.index[0]

print(f"Top sales category: {best_category}")
print(f"Top sales region: {best_region}")
print(f"Top sales customer segment: {best_segment}")